# Amazon Reviews 2023 → Zilliz Cloud: multimodal ingest

Builds a Milvus/Zilliz collection from **Amazon Reviews 2023** item metadata (McAuley Lab) for a query-understanding + filtering demo.

For each product we store:
- **`image_vec`** — SigLIP embedding of the *large* product image (`ViT-L-16-SigLIP-256`, 1024-d). Enables image→image and SigLIP text→image.
- **`text_vec`** — embedding of a structured `title + brand + category + features + description` string (default `Qwen3-Embedding-0.6B`, 1024-d, swappable).
- Filterable scalars: `price`, `average_rating`, `rating_number`, `main_category`, `store` (brand), `categories[]`, plus a dynamic field for stray attributes (colour/material).

**Run on a T4 GPU runtime.** Network (image download) is the bottleneck, not the GPU. Drive checkpointing makes the run resumable. Validate with a smaller `TARGET_N` (e.g. 5k) before scaling to 100k. 100k × 2 × 1024-d float vectors is ~0.8 GB of raw vectors before index/payload, so check your cluster capacity in the console.

In [18]:
# 1. Install deps  (no -U: don't clobber Colab's pinned pillow / requests / tqdm)
!pip install -q pymilvus "open_clip_torch>=2.24" "sentence-transformers>=3.0" \
    "transformers>=4.51" "datasets>=2.19" einops

In [19]:
# 2. Imports + config
import os, io, re, json, time
import numpy as np
import torch
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ---- What to ingest --------------------------------------------------------
HF_CONFIG       = "raw_meta_Electronics"     # any raw_meta_<Category> config
COLLECTION_NAME = "amazon_reviews_electronics"
TARGET_N        = 100_000                     # products to ingest (start smaller to validate)

# ---- Models ----------------------------------------------------------------
SIGLIP_NAME     = "ViT-L-16-SigLIP-256"
SIGLIP_PRETRAIN = "webli"
TEXT_MODEL      = "Qwen/Qwen3-Embedding-0.6B" # swap -> "BAAI/bge-m3" for dense+sparse later

# ---- Batch / network tuning ------------------------------------------------
FETCH_BATCH = 256     # rows buffered before a download+embed+upsert flush
DL_WORKERS  = 10      # concurrent image downloads
IMG_BATCH   = 64      # SigLIP image batch on GPU
TXT_BATCH   = 64      # text embedding batch
CONTACT_UA  = "ZillizDemo/1.0 (simon@zilliz.com)"  # polite User-Agent

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---- Zilliz credentials (use Colab Secrets; never hardcode) ----------------
try:
    from google.colab import userdata
    ZILLIZ_URI   = userdata.get("ZILLIZ_URI")
    ZILLIZ_TOKEN = userdata.get("ZILLIZ_TOKEN")
except Exception:
    import getpass
    ZILLIZ_URI   = getpass.getpass("Zilliz public endpoint URI: ")
    ZILLIZ_TOKEN = getpass.getpass("Zilliz API token: ")
assert ZILLIZ_URI and ZILLIZ_TOKEN, "Set ZILLIZ_URI / ZILLIZ_TOKEN"

device: cuda


In [20]:
# 3. Mount Drive for resumable checkpoints
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/amazon_ingest"
except Exception:
    CKPT_DIR = "./amazon_ingest"   # local fallback
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, f"{COLLECTION_NAME}.ckpt.json")

def load_ckpt():
    if os.path.exists(CKPT_PATH):
        with open(CKPT_PATH) as f:
            return json.load(f)
    return {"processed": 0, "inserted": 0}

def save_ckpt(ck):
    tmp = CKPT_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(ck, f)
    os.replace(tmp, CKPT_PATH)

print("checkpoint:", CKPT_PATH, "->", load_ckpt())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint: /content/drive/MyDrive/amazon_ingest/amazon_reviews_electronics.ckpt.json -> {'processed': 18554, 'inserted': 10239}


In [21]:
# 4. Cleaning / parsing helpers
TAG_RE = re.compile(r"<[^>]+>")
WS_RE  = re.compile(r"\s+")

def clean(s):
    if not s:
        return ""
    s = TAG_RE.sub(" ", str(s))
    return WS_RE.sub(" ", s).strip()

def trunc_bytes(s, max_bytes):
    # Zilliz enforces VARCHAR max_length in UTF-8 BYTES, not characters.
    if s is None:
        return ""
    return str(s).encode("utf-8")[:max_bytes].decode("utf-8", errors="ignore")

def parse_price(p):
    if p is None:
        return -1.0
    if isinstance(p, (int, float)):
        return float(p) if (p == p and p > 0) else -1.0
    m = re.search(r"\d+(?:\.\d+)?", str(p).replace(",", ""))
    return float(m.group()) if m else -1.0

def pick_large_image(images):
    # raw_meta `images` is usually {hi_res:[...], large:[...], thumb:[...], variant:[...]}
    if not images:
        return None
    if isinstance(images, dict):
        for key in ("large", "hi_res", "thumb"):
            arr = images.get(key) or []
            arr = [arr] if isinstance(arr, str) else arr
            for u in arr:
                if u:
                    return u
    elif isinstance(images, list):
        for im in images:
            if isinstance(im, dict):
                for key in ("large", "hi_res", "thumb"):
                    if im.get(key):
                        return im[key]
    return None

def build_text(rec):
    parts = []
    title = clean(rec.get("title"))
    if title:
        parts.append(title)
    store = clean(rec.get("store"))
    if store:
        parts.append("Brand: " + store)
    cats = [c for c in (rec.get("categories") or []) if c]
    if cats:
        parts.append("Category: " + " > ".join(cats[:6]))
    feats = [clean(f) for f in (rec.get("features") or []) if f]
    if feats:
        parts.append(" ".join(feats[:12]))
    desc = " ".join(clean(d) for d in (rec.get("description") or []) if d)
    if desc:
        parts.append(desc[:1500])           # keep the front; title/features already added
    return "\n".join(parts).strip()

def display_snippet(rec):
    title = clean(rec.get("title"))
    desc  = " ".join(clean(d) for d in (rec.get("description") or []) if d)
    return (title + (" — " + desc if desc else "")).strip()

def extract_attrs(rec):
    out = {}
    d = rec.get("details") or {}
    if isinstance(d, dict):
        for keys, name in [(("Color", "Colour", "color"), "color"),
                           (("Material", "material"), "material")]:
            for k in keys:
                if d.get(k):
                    out[name] = trunc_bytes(clean(d[k]), 120)
                    break
    return out

In [22]:
# 5. Load embedding models
import open_clip
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    SIGLIP_NAME, pretrained=SIGLIP_PRETRAIN)
clip_tokenizer = open_clip.get_tokenizer(SIGLIP_NAME)
clip_model = clip_model.to(device).eval()

text_model = SentenceTransformer(TEXT_MODEL, device=device, trust_remote_code=True)

# Probe dims so the schema is never hardcoded wrong.
with torch.no_grad():
    _probe = clip_preprocess(Image.new("RGB", (256, 256))).unsqueeze(0).to(device)
    IMG_DIM = int(clip_model.encode_image(_probe).shape[-1])
TXT_DIM = int(text_model.get_sentence_embedding_dimension())
print(f"IMG_DIM={IMG_DIM}  TXT_DIM={TXT_DIM}")

@torch.no_grad()
def embed_images(pil_list):
    out = []
    for i in range(0, len(pil_list), IMG_BATCH):
        chunk = pil_list[i:i + IMG_BATCH]
        t = torch.stack([clip_preprocess(im) for im in chunk]).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            f = clip_model.encode_image(t)
        out.append(F.normalize(f, dim=-1).float().cpu().numpy())
    return np.concatenate(out, axis=0)

def embed_texts(texts):
    return text_model.encode(texts, batch_size=TXT_BATCH,
                             normalize_embeddings=True, convert_to_numpy=True,
                             show_progress_bar=False)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

IMG_DIM=1024  TXT_DIM=1024


/tmp/ipykernel_570/4273600960.py:17: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  TXT_DIM = int(text_model.get_sentence_embedding_dimension())


In [23]:
# 6. Connect to Zilliz, define schema, create collection
from pymilvus import MilvusClient, DataType

client = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)

if client.has_collection(COLLECTION_NAME):
    print("Collection exists. Set DROP=True to recreate.")
    DROP = False
    if DROP:
        client.drop_collection(COLLECTION_NAME)

if not client.has_collection(COLLECTION_NAME):
    schema = client.create_schema(auto_id=False, enable_dynamic_field=True)
    schema.add_field("parent_asin",    DataType.VARCHAR, is_primary=True, max_length=24)
    schema.add_field("title",          DataType.VARCHAR, max_length=1024)
    schema.add_field("main_category",  DataType.VARCHAR, max_length=128)
    schema.add_field("store",          DataType.VARCHAR, max_length=256)
    schema.add_field("price",          DataType.FLOAT)        # -1.0 = unknown
    schema.add_field("average_rating", DataType.FLOAT)        # -1.0 = unknown
    schema.add_field("rating_number",  DataType.INT64)        # 0    = unknown
    schema.add_field("categories",     DataType.ARRAY, element_type=DataType.VARCHAR,
                     max_capacity=16, max_length=128)
    schema.add_field("image_url",      DataType.VARCHAR, max_length=1024)
    schema.add_field("text_snippet",   DataType.VARCHAR, max_length=2048)
    schema.add_field("image_vec",      DataType.FLOAT_VECTOR, dim=IMG_DIM)
    schema.add_field("text_vec",       DataType.FLOAT_VECTOR, dim=TXT_DIM)

    idx = client.prepare_index_params()
    idx.add_index(field_name="image_vec", index_type="AUTOINDEX", metric_type="COSINE")
    idx.add_index(field_name="text_vec",  index_type="AUTOINDEX", metric_type="COSINE")
    # Scalar indexes for fast filtering. If Serverless rejects these, delete the
    # four lines below — AUTOINDEX still auto-handles scalar filtering.
    for fld in ("price", "average_rating", "main_category", "store"):
        idx.add_index(field_name=fld, index_type="AUTOINDEX")

    client.create_collection(COLLECTION_NAME, schema=schema, index_params=idx)
    print("Created", COLLECTION_NAME)

print(client.describe_collection(COLLECTION_NAME)["collection_name"], "ready")

Collection exists. Set DROP=True to recreate.
amazon_reviews_electronics ready


In [ ]:
# 7. Stream raw meta_<CATEGORY>.jsonl.gz directly and ingest (resumable)
import gzip, os, io, time, json, http.client
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import urllib3

text_model.max_seq_length = 384   # caps the quadratic attention cost
TXT_BATCH = 32
DL_WORKERS = 10
import torch; torch.cuda.empty_cache()

CATEGORY = "Electronics"   # change if you're ingesting a different category
META_URL = ("https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/"
            f"raw/meta_categories/meta_{CATEGORY}.jsonl.gz")
LOCAL_GZ = f"/content/meta_{CATEGORY}.jsonl.gz"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": CONTACT_UA})

NET_ERRORS = (requests.exceptions.RequestException,
              urllib3.exceptions.HTTPError,
              http.client.HTTPException)

def download_resumable(url, dest, session, chunk=1 << 20, max_retries=100):
    """Append compressed bytes to disk, resuming with HTTP Range on drop."""
    head = session.head(url, timeout=30, allow_redirects=True)
    total = int(head.headers.get("Content-Length") or 0) or None
    ranges_ok = head.headers.get("Accept-Ranges", "").lower() == "bytes"

    attempt = 0
    while True:
        have = os.path.getsize(dest) if os.path.exists(dest) else 0
        if total and have >= total:          # already complete (e.g. on re-run)
            return dest
        headers, mode = {}, "wb"
        if have and ranges_ok:
            headers["Range"], mode = f"bytes={have}-", "ab"
        elif have:                            # server can't resume -> restart
            have, mode = 0, "wb"
        try:
            with session.get(url, headers=headers, stream=True, timeout=120) as r:
                if "Range" in headers and r.status_code == 200:   # range ignored
                    have, mode = 0, "wb"
                r.raise_for_status()
                if total is None:
                    cl = int(r.headers.get("Content-Length") or 0)
                    total = have + cl if cl else None
                with open(dest, mode) as f, tqdm(
                        total=total, initial=have, unit="B", unit_scale=True,
                        unit_divisor=1024, desc=f"download {CATEGORY}") as dbar:
                    for part in r.iter_content(chunk_size=chunk):
                        if part:
                            f.write(part); dbar.update(len(part))
            return dest                        # iter_content exhausted -> done
        except NET_ERRORS:
            attempt += 1
            if attempt > max_retries:
                raise
            time.sleep(min(2 ** attempt, 30))  # backoff, then resume from `have`

def fetch_image(url, retries=3, timeout=12):
    for i in range(retries):
        try:
            r = SESSION.get(url, timeout=timeout)
            if r.status_code == 200:
                return Image.open(io.BytesIO(r.content)).convert("RGB")
            if r.status_code in (429, 503):
                time.sleep(2 * (i + 1))
        except Exception:
            time.sleep(1.5 * (i + 1))
    return None

def make_row(rec, asin, url, text, snippet, iv, tv):
    row = {
        "parent_asin":    trunc_bytes(asin, 24),
        "title":          trunc_bytes(clean(rec.get("title")), 1020),
        "main_category":  trunc_bytes(clean(rec.get("main_category")), 124),
        "store":          trunc_bytes(clean(rec.get("store")), 252),
        "price":          parse_price(rec.get("price")),
        "average_rating": float(rec.get("average_rating") or -1.0),
        "rating_number":  int(rec.get("rating_number") or 0),
        "categories":     [trunc_bytes(clean(c), 124) for c in (rec.get("categories") or [])[:16] if c],
        "image_url":      trunc_bytes(url, 1020),
        "text_snippet":   trunc_bytes(snippet, 2040),
        "image_vec":      iv.tolist(),
        "text_vec":       tv.tolist(),
    }
    row.update(extract_attrs(rec))
    return row

def flush(buffer):
    imgs = list(ThreadPoolExecutor(max_workers=DL_WORKERS).map(
        fetch_image, [b["url"] for b in buffer]))
    keep = [(b, im) for b, im in zip(buffer, imgs) if im is not None]
    if not keep:
        return 0
    iv = embed_images([im for _, im in keep])
    tv = embed_texts([b["text"] for b, _ in keep])
    rows = [make_row(b["rec"], b["asin"], b["url"], b["text"], b["snippet"], iv[i], tv[i])
            for i, (b, _) in enumerate(keep)]
    client.upsert(COLLECTION_NAME, rows)
    return len(rows)

def stream_meta_local(path):
    with gzip.open(path, "rb") as gz:
        for line in gz:
            if line.strip():
                yield json.loads(line)

download_resumable(META_URL, LOCAL_GZ, SESSION)   # fetch compressed bytes to disk first

ck   = load_ckpt()
skip = ck["processed"]
pbar = tqdm(total=TARGET_N, initial=ck["inserted"], desc="ingested")
buffer, seen, idx = [], set(), 0
try:
    for rec in stream_meta_local(LOCAL_GZ):
        idx += 1
        if idx <= skip:
            continue
        ck["processed"] = idx
        if ck["inserted"] >= TARGET_N:
            break
        asin = rec.get("parent_asin")
        if not asin or asin in seen:
            continue
        url  = pick_large_image(rec.get("images"))
        text = build_text(rec)
        price = parse_price(rec.get("price"))
        if not url or not text or price < 0:   # require image, text, and a known price
            continue
        seen.add(asin)
        buffer.append({"rec": rec, "asin": asin, "url": url,
                       "text": text, "snippet": display_snippet(rec)})
        if len(buffer) >= FETCH_BATCH:
            n = flush(buffer); buffer = []
            ck["inserted"] += n; pbar.update(n); save_ckpt(ck)
    if buffer:
        n = flush(buffer); buffer = []
        ck["inserted"] += n; pbar.update(n); save_ckpt(ck)
finally:
    pbar.close()
print("done. inserted:", ck["inserted"], "scanned:", ck["processed"])

download Electronics:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

In [ ]:
# 8. Finalize: load + count
client.load_collection(COLLECTION_NAME)
print("entities:", client.get_collection_stats(COLLECTION_NAME)["row_count"])

## Sanity checks
Text+filter search (the core demo motion), image→image similarity, and SigLIP text→image cross-modal.

In [ ]:
# 9a. Natural-language text search + scalar filters
query = "wireless noise cancelling over-ear headphones"
task  = "Given a shopping query, retrieve relevant product listings"
# Qwen3 uses an instruction on the QUERY side only; documents were embedded plain.
qv = text_model.encode([query], prompt=f"Instruct: {task}\nQuery: ",
                       normalize_embeddings=True, convert_to_numpy=True)

res = client.search(
    COLLECTION_NAME, data=qv, anns_field="text_vec", limit=5,
    filter="price > 0 and price < 120 and average_rating >= 4.0",
    output_fields=["title", "store", "price", "average_rating", "image_url"])
for hit in res[0]:
    e = hit["entity"]
    print(f"{hit['distance']:.3f}  £{e['price']:.2f}  {e['average_rating']}*  {e['title'][:80]}")

In [ ]:
# 9b. Image -> image (visually similar)
sample = client.query(COLLECTION_NAME, filter="price > 0",
                      output_fields=["image_vec", "title"], limit=1)
res = client.search(COLLECTION_NAME, data=[sample[0]["image_vec"]],
                    anns_field="image_vec", limit=5,
                    output_fields=["title", "image_url"])
print("seed:", sample[0]["title"][:80])
for hit in res[0]:
    print(f"  {hit['distance']:.3f}  {hit['entity']['title'][:80]}")

In [ ]:
# 9c. SigLIP text -> image (cross-modal, against image_vec)
with torch.no_grad():
    toks = clip_tokenizer(["a red leather handbag"]).to(device)
    tf = F.normalize(clip_model.encode_text(toks), dim=-1).cpu().numpy()
res = client.search(COLLECTION_NAME, data=tf, anns_field="image_vec", limit=5,
                    output_fields=["title", "image_url"])
for hit in res[0]:
    print(f"{hit['distance']:.3f}  {hit['entity']['title'][:80]}")

## Next
Once this is populated, the Gradio app adds the query-understanding layer: an LLM parses a natural-language query into the scalar `filter` string (price / rating / brand / category) plus the residual semantic text, then runs text→text (and optionally text→image) search with those filters. Swap `TEXT_MODEL` to `BAAI/bge-m3` if you want to add a sparse vector and show dense+sparse+scalar hybrid.